# A visual walkthrough

This notebook illustrates the ideas in the [architecture overview](../explanation/architecture)
and the [mathematical foundations](../explanation/mathematical-foundations) on one small,
abstract problem: locating a source in 2D from noisy distance measurements to a few sensors.
The problem is deliberately simple so the *ideas* stay in view, but every step uses
photomancy's real engine, the same `Fit`, `run`, `Posterior`, `cluster_to_mixture`, and
expected information gain you would use on an orbit or an atmosphere.

With two sensors the problem is ambiguous: a source and its mirror image across the sensor
line fit the data equally well. That ambiguity is the thread that ties the walkthrough
together.

## The setup

Three pieces describe the problem. The scene is a `Source` holding one array, the 2D
position. The forward model turns a scene into the distances a sensor would measure. The
data are those distances for two sensors on the x-axis, with Gaussian noise of 0.25 added.
The forward model is a small `equinox` module so that the sensor positions it holds stay
arrays the engine can trace.

In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse
from priorities import Uniform

from photomancy.infer import (
    Fit,
    ForwardTerm,
    Laplace,
    cluster_to_mixture,
    run,
    weighted_cov,
    weighted_mean,
)
from photomancy.infer.eig import evaluate_candidates
from photomancy.infer.likelihoods import gaussian_loglike

jax.config.update("jax_enable_x64", True)


class Source(eqx.Module):
    # the scene: a 2D source position (the only fitted leaf)
    pos: jnp.ndarray


class Distances(eqx.Module):
    # the forward model: distance from the source to each sensor
    sensors: jnp.ndarray

    def __call__(self, scene):
        return jnp.linalg.norm(scene.pos - self.sensors, axis=1)


sensors = jnp.array([[-3.0, 0.0], [3.0, 0.0]])  # two sensors on the x-axis
true_pos = jnp.array([1.0, 2.2])
sigma = 0.25
obs = Distances(sensors)(Source(true_pos)) + sigma * jax.random.normal(
    jax.random.key(1), (sensors.shape[0],)
)
print("measured distances:", np.round(np.array(obs), 2).tolist())

## 1. The fit

A `Fit` is a log density over the free entries of a scene. It holds three things: the base
scene, a law for each entry that is free (here a broad box for the position), and the data
terms. A `ForwardTerm` is the generic data term: the forward model, and a Gaussian
likelihood that scores its output against the measured distances.

Evaluating the fit's log density on a grid shows the posterior landscape: two equally good
modes, the true source and its mirror across the sensor line.

In [ ]:
box = Uniform(low=jnp.array([-5.0, -5.0]), high=jnp.array([5.0, 5.0]))
term = ForwardTerm(Distances(sensors), gaussian_loglike(obs, sigma))
fit = Fit(Source(jnp.zeros(2)), {"pos": box}, (term,))


def grid_density(fit, lim=5.0, n=160):
    # the fit's normalized density on an n x n grid of positions
    xs = jnp.linspace(-lim, lim, n)
    X, Y = jnp.meshgrid(xs, xs)
    xy = jnp.stack([X.ravel(), Y.ravel()], axis=1)
    lp = fit.log_density({"pos": xy}).reshape(n, n)
    return np.array(X), np.array(Y), np.array(jnp.exp(lp - lp.max()))


def draw_ellipse(ax, mean, cov, **kw):
    # the 2-sigma ellipse of a Gaussian
    vals, vecs = jnp.linalg.eigh(cov)
    ang = float(jnp.degrees(jnp.arctan2(vecs[1, 0], vecs[0, 0])))
    w, h = 2 * 2.0 * jnp.sqrt(jnp.maximum(vals, 0))
    ax.add_patch(
        Ellipse(np.array(mean), float(w), float(h), angle=ang, fill=False, **kw)
    )


X, Y, P = grid_density(fit)
fig, ax = plt.subplots(figsize=(5, 5))
ax.contourf(X, Y, P, levels=30, cmap="magma")
ax.scatter(*np.array(sensors).T, marker="^", c="cyan", s=80, label="sensors")
ax.scatter(*np.array(true_pos), marker="*", c="white", s=160, label="true source")
ax.set_title("posterior landscape (two sensors)")
ax.legend()
plt.show()

## 2. The Laplace approximation

The Laplace approximation finds a mode, then fits a Gaussian from the curvature there.
`run(fit, backend, key)` is the one call that turns a fit into a `Posterior`: a cloud of
draws with weights, together with the log evidence. The `Laplace` backend draws from its
Gaussian and weights the draws back to the true density. Here it starts at one explicit
position, near the true source.

A single start finds a single mode, so one Gaussian cannot represent the ambiguity on its
own. The mean and covariance below are the weighted mean and covariance of the cloud.

In [ ]:
single = run(
    fit,
    Laplace(n_draws=4000, n_steps=200, init={"pos": jnp.array([1.0, 2.0])}),
    jax.random.key(0),
)
draws = single.samples["pos"]
mean = weighted_mean(single.log_weights, draws)
cov = weighted_cov(single.log_weights, draws)
print("mean:", np.round(np.array(mean), 2).tolist())
print(f"log evidence: {float(single.log_evidence):.2f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.contourf(X, Y, P, levels=30, cmap="magma")
ax.scatter(*np.array(mean), c="cyan", s=60, label="posterior mean")
draw_ellipse(ax, mean, cov, edgecolor="cyan", lw=2)
ax.set_title("Laplace Gaussian (one mode)")
ax.legend()
plt.show()

## 3. A mixture over the modes

The same backend takes several starts at once. A start dictionary whose entries carry a
leading axis of length `n_modes` gives each mode its own start, so one start per basin
recovers both modes, and the draws are weighted by each mode's evidence. The result is
still one `Posterior`, now a cloud with two clumps.

`cluster_to_mixture` summarizes that cloud as a Gaussian mixture: it clusters the draws into
`k` modes, around the modes `Laplace` found (`info.modes`, usable as `init`), and returns a `priorities.Mixture` with one mean, covariance and weight per
cluster. Because the two sensors lie on a line, the likelihood is exactly symmetric across
it, so the two modes carry equal weight.

In [ ]:
starts = {"pos": jnp.array([[1.5, 2.0], [1.5, -2.0]])}
both = run(
    fit,
    Laplace(n_draws=4000, n_modes=2, n_steps=200, init=starts),
    jax.random.key(0),
)
mix = cluster_to_mixture(both, 2, jax.random.key(1), init=both.info.modes)
means = mix.component_distribution.loc
tril = mix.component_distribution.scale_tril
covs = tril @ jnp.swapaxes(tril, -1, -2)

print("modes:", np.round(np.array(means), 2).tolist())
print("weights:", np.round(np.array(jax.nn.softmax(mix.mixing_logits)), 3).tolist())
print(f"total log evidence: {float(both.log_evidence):.2f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.contourf(X, Y, P, levels=30, cmap="magma")
for k in range(2):
    draw_ellipse(ax, means[k], covs[k], edgecolor="cyan", lw=1.5)
ax.set_title("mixture: both modes")
plt.show()

## 4. The Bayes factor between the modes

Treating the two modes as competing hypotheses, the ratio of the posterior mass each holds is
a Bayes factor, because the prior gives the two sides of the sensor line equal mass. The
weights of the cloud give that mass directly: the mass above the sensor line against the mass
below it. With symmetric data the ratio is one, so the log Bayes factor is near zero: the
data alone cannot decide between the source and its mirror. This is the quantity an
observation has to move.

In [ ]:
w = jax.nn.softmax(both.log_weights)
mass_above = float(jnp.sum(w * (both.samples["pos"][:, 1] > 0)))
log_bayes_factor = np.log(mass_above / (1 - mass_above))
print(f"mass above / below the line: {mass_above:.3f} / {1 - mass_above:.3f}")
print("log Bayes factor (above vs below):", round(float(log_bayes_factor), 2))

## 5. Where to look next: expected information gain

The expected information gain scores a candidate observation by how much it is expected to
sharpen the belief. `evaluate_candidates` takes the mixture and a forward model for the
candidate, here the distance from the source to a candidate sensor position, and returns the
EIG over a grid of placements. The mixture lives on the flat vector of the fitted entries,
which here is the position itself, so the forward model takes the position `z` and the
candidate `c` directly.

A sensor on the baseline cannot tell the two mirror modes apart, so the EIG is near zero
there; the most informative placements sit off the baseline, where the two modes predict very
different distances.

In [ ]:
grid = np.stack(
    np.meshgrid(np.linspace(-4, 4, 25), np.linspace(-4, 4, 25)), -1
).reshape(-1, 2)
candidates = jnp.array(grid)


def candidate_forward(z, c):
    return jnp.atleast_1d(jnp.linalg.norm(z - c))


eig = evaluate_candidates(mix, candidates, candidate_forward, sigma**2)
best = np.array(candidates[int(jnp.argmax(eig["total_eig"]))])
print("best next sensor:", np.round(best, 2).tolist())

fig, ax = plt.subplots(figsize=(5, 5))
ax.tricontourf(
    grid[:, 0], grid[:, 1], np.array(eig["total_eig"]), levels=30, cmap="viridis"
)
ax.scatter(*np.array(sensors).T, marker="^", c="white", s=80)
ax.scatter(*best, marker="X", c="red", s=140, label="best next sensor")
ax.set_title("expected information gain over candidate sensors")
ax.legend()
plt.show()

## 6. Observing and updating

Placing a sensor at the most informative spot and folding its measurement into the belief is
sequential Bayesian updating. The mixture becomes the law of the position in the next `Fit`,
in place of the broad box, and the new measurement is its only data term: the old data are
already inside the mixture. The new measurement is consistent with one mode and badly
inconsistent with its mirror, so the belief collapses onto the true source.

The fit starts from both modes, so the backend has to find that one of them is ruled out.

In [ ]:
new_sensor = jnp.array(best)[None]
new_obs = Distances(new_sensor)(Source(true_pos)) + sigma * jax.random.normal(
    jax.random.key(2), (1,)
)
next_fit = Fit(
    Source(jnp.zeros(2)),
    {"pos": mix},
    (ForwardTerm(Distances(new_sensor), gaussian_loglike(new_obs, sigma)),),
)
collapsed = run(
    next_fit,
    Laplace(n_draws=4000, n_modes=2, n_steps=300, init={"pos": means}),
    jax.random.key(3),
)

w = jax.nn.softmax(collapsed.log_weights)
mass_above = float(jnp.sum(w * (collapsed.samples["pos"][:, 1] > 0)))
mean = weighted_mean(collapsed.log_weights, collapsed.samples["pos"])
print(f"posterior mass above the line: {mass_above:.4f}")
print("collapsed posterior mean:", np.round(np.array(mean), 2).tolist())

Xc, Yc, Pc = grid_density(next_fit)
fig, ax = plt.subplots(figsize=(5, 5))
ax.contourf(Xc, Yc, Pc, levels=30, cmap="magma")
ax.scatter(*np.array(true_pos), marker="*", c="white", s=160, label="true source")
ax.scatter(*np.array(new_sensor[0]), marker="^", c="red", s=80, label="new sensor")
ax.set_title("the belief collapses to one mode")
ax.legend()
plt.show()

## What this shows

Five objects carried the whole walkthrough: a `Fit` over a scene, a backend that `run` turns
into a `Posterior`, a mixture that holds the ambiguity, the evidence that scores competing
explanations, and the expected information gain that decides where to look next. None of it
was specific to source localization. Swapping the scene and the forward model for an orbit
or an atmosphere reuses every step unchanged, which is the point of the
[architecture](../explanation/architecture).